# Document Extraction POC

This is the main notebook to run. The first code cell loads the helper notebooks, then the remaining cells walk through one document from upload to saved JSON.


In [17]:
# Load helper notebooks
# These notebooks define the functions used below, so run this cell first.
# After this, the main POC cells can call detect_file_type(), extract_text(), extract_fields(), and save_json().
from pathlib import Path
cwd = Path.cwd()
if (cwd / "01_project_setup.py").exists():
    notebooks_dir = cwd
elif (cwd / "notebooks" / "01_project_setup.py").exists():
    notebooks_dir = cwd / "notebooks"
elif (cwd.parent / "notebooks" / "01_project_setup.py").exists():
    notebooks_dir = cwd.parent / "notebooks"
else:
    raise FileNotFoundError(f"Cannot find helper Python files from cwd={cwd}")

notebook_path = notebooks_dir / "01_project_setup.py"
%run $notebook_path
notebook_path = notebooks_dir / "02_text_extraction.py"
%run $notebook_path
notebook_path = notebooks_dir / "03_field_extraction.py"
%run $notebook_path
notebook_path = notebooks_dir / "04_save_output.py"
%run $notebook_path
notebook_path = notebooks_dir / "05_validation.py"
%run $notebook_path


Upload folder: /Users/chandrapratap/document-extraction-system/uploads
Output folder: /Users/chandrapratap/document-extraction-system/outputs


In [9]:
# Choose the input file
# Type the name of a file that is already inside the uploads folder.
# Supported examples: AnjaliSharma.pdf, NehaPatil.png, or EmployeeForm.docx.

file_input = widgets.Text(
    value="",
    placeholder="Enter uploaded file name",
    description="File:",
    disabled=False
)

display(file_input)


Text(value='', description='File:', placeholder='Enter uploaded file name')

In [10]:
# Check the selected file
# This confirms the file exists inside uploads and then locks the text box so the same file is used for the rest of the run.

file_path = get_file_path(file_input.value)
file_input.disabled = True

print("File selected:", file_path)


File selected: /Users/chandrapratap/document-extraction-system/uploads/AnjaliSharma.pdf


In [11]:
# Detect the file type
# The next cell needs to know whether this is a PDF, image, or DOCX file.

file_type = detect_file_type(file_path)

print("Detected file type:", file_type)


Detected file type: pdf


In [12]:
# Read text from the document
# This is the raw text found in the file. If the JSON looks wrong later, start by checking this output.

final_text = extract_text(file_path, file_type)

print(final_text)


EMPLOYEE ENROLLMENT FORM
Field Value
Employee Name Anjali Sharma
Employee ID EMP-2045
Date of Birth 14/08/1997
Date of Joining 10/01/2026
Department Human Resources
Designation HR Associate
Mobile Number 9876543210
Email anjali.sharma@example.com
Address 12 MG Road, Bengaluru
Pincode 560001
Signature: ____________________


In [15]:
# Convert text into JSON fields
# This searches the raw text for the form labels and returns a dictionary with one value per expected field.

extracted_json = extract_fields(final_text, form_schema)

print(json.dumps(extracted_json, indent=4, ensure_ascii=False))


{
    "employee_name": "Anjali Sharma",
    "employee_id": "EMP-2045",
    "date_of_birth": "14/08/1997",
    "date_of_joining": "10/01/2026",
    "department": "Human Resources",
    "designation": "HR Associate",
    "mobile_number": "9876543210",
    "email": "anjali.sharma@example.com",
    "address": "12 MG Road, Bengaluru",
    "pincode": "560001"
}


In [18]:
# Validate extracted fields and save the JSON file
# The validation step produces a structured output containing the extracted data,
# validation results per field, and a validation summary. That structure is then
# saved to the outputs folder with a timestamp.

validated_output = validate_extracted_fields(extracted_json)

json_output_path = save_json(file_path, validated_output, outputs_folder)

print("JSON output saved:", json_output_path)


JSON output saved: /Users/chandrapratap/document-extraction-system/outputs/AnjaliSharma_20260608_114943.json
